# MANOVA and MANCOVA: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.multivariate.manova import MANOVA
import statsmodels.formula.api as smf

rng = np.random.default_rng(9)

### 1. Three groups, two correlated outcomes; the shifts go against the correlation

In [ ]:
# 1. Three groups, two correlated outcomes; the shifts go against the correlation
n, rho, shift = 30, 0.8, 0.45
cov = np.array([[1, rho], [rho, 1]])
shifts = [(1, -1), (0, 0), (-1, 1)]
rows = []
for g, d in enumerate(shifts):
    Y = rng.multivariate_normal([shift * d[0], shift * d[1]], cov, n)
    for a, b in Y:
        rows.append((g + 1, a, b))
df = pd.DataFrame(rows, columns=["g", "y1", "y2"])
print(df.groupby("g")[["y1", "y2"]].mean().round(2))

### 2. Univariate ANOVA on each outcome

In [ ]:
# 2. Univariate ANOVA on each outcome
for y in ("y1", "y2"):
    print(y, "ANOVA p =", format(stats.f_oneway(*[d[y] for _, d in df.groupby("g")]).pvalue, ".4f"))

### 3. MANOVA with statsmodels

In [ ]:
# 3. MANOVA with statsmodels
mv = MANOVA.from_formula("y1 + y2 ~ C(g)", data=df).mv_test()
tab = mv.results["C(g)"]["stat"]
print("\nstatsmodels Wilks lambda:", round(tab.loc["Wilks' lambda", "Value"], 4), "| F =", round(tab.loc["Wilks' lambda", "F Value"], 3), "| p =", format(tab.loc["Wilks\' lambda", "Pr > F"], ".2e"))

### 4. The same statistic from the SSCP matrices (W, B, T)

In [ ]:
# 4. The same statistic from the SSCP matrices (W, B, T)
Y = df[["y1", "y2"]].values; gm = Y.mean(0)
W = sum(((d[["y1", "y2"]].values - d[["y1", "y2"]].values.mean(0)).T @ (d[["y1", "y2"]].values - d[["y1", "y2"]].values.mean(0))) for _, d in df.groupby("g"))
B = sum(len(d) * np.outer(d[["y1", "y2"]].values.mean(0) - gm, d[["y1", "y2"]].values.mean(0) - gm) for _, d in df.groupby("g"))
T = W + B
lam = np.linalg.det(W) / np.linalg.det(T)
N, g = len(df), 3
Fv = (1 - np.sqrt(lam)) / np.sqrt(lam) * (N - g - 1) / (g - 1)
print("by hand: Lambda =", round(lam, 4), "| Rao F(", 2 * (g - 1), ",", 2 * (N - g - 1), ") =", round(Fv, 3), "| p =", format(stats.f.sf(Fv, 2 * (g - 1), 2 * (N - g - 1)), ".2e"))

### 5. Two groups: Wilks lambda equals Hotelling's T^2 test

In [ ]:
# 5. Two groups: Wilks lambda equals Hotelling's T^2 test
two = df[df.g != 2]
a, b = two[two.g == 1][["y1", "y2"]].values, two[two.g == 3][["y1", "y2"]].values
d = a.mean(0) - b.mean(0)
Sp = ((len(a) - 1) * np.cov(a.T) + (len(b) - 1) * np.cov(b.T)) / (len(a) + len(b) - 2)
T2 = len(a) * len(b) / (len(a) + len(b)) * d @ np.linalg.inv(Sp) @ d
nn = len(a) + len(b)
F2 = (nn - 3) / (2 * (nn - 2)) * T2
print("\nHotelling T2 =", round(T2, 3), "| F =", round(F2, 3), "| p =", format(stats.f.sf(F2, 2, nn - 3), ".2e"), "| Lambda = 1/(1 + T2/(N-2)) =", round(1 / (1 + T2 / (nn - 2)), 4))

### 6. MANCOVA: add a covariate that drifts across groups

In [ ]:
# 6. MANCOVA: add a covariate that drifts across groups
df["z"] = df.g.map({1: -0.8, 2: 0.0, 3: 0.8}) + rng.standard_normal(len(df))
df["y1"] += 0.8 * df.z; df["y2"] += 0.8 * df.z
m0 = MANOVA.from_formula("y1 + y2 ~ C(g)", data=df).mv_test().results["C(g)"]["stat"]
m1 = MANOVA.from_formula("y1 + y2 ~ C(g) + z", data=df).mv_test().results["C(g)"]["stat"]
print("\nWilks lambda without the covariate:", round(m0.loc["Wilks' lambda", "Value"], 4), "p =", format(m0.loc["Wilks\' lambda", "Pr > F"], ".2e"))
print("Wilks lambda adjusting for z (MANCOVA):", round(m1.loc["Wilks' lambda", "Value"], 4), "p =", format(m1.loc["Wilks\' lambda", "Pr > F"], ".2e"))
print("covariate effect on y1: slope", round(smf.ols("y1 ~ C(g) + z", df).fit().params["z"], 2), "(true 0.8)")

## Expected output

Running all cells above should print:

```
     y1    y2
g            
1  0.73 -0.07
2 -0.09  0.12
3 -0.55  0.39
y1 ANOVA p = 0.0000
y2 ANOVA p = 0.1941

statsmodels Wilks lambda: 0.4037 | F = 24.679 | p = 3.71e-16
by hand: Lambda = 0.4037 | Rao F( 4 , 172 ) = 24.679 | p = 3.71e-16

Hotelling T2 = 131.545 | F = 64.638 | p = 2.20e-15 | Lambda = 1/(1 + T2/(N-2)) = 0.306

Wilks lambda without the covariate: 0.4607 p = 9.60e-14
Wilks lambda adjusting for z (MANCOVA): 0.4989 p = 3.81e-12
covariate effect on y1: slope 0.77 (true 0.8)
```

## Practice exercises

Try these in the cells below.

1. With no group difference, how often is the MANOVA significant over repeated samples (about 5%)?
2. Make the groups differ only in the second outcome. Compare MANOVA with the univariate ANOVA p-values.
3. Switch on the covariate that differs by group and compare Λ with and without adjustment.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
